# ClinIQ: fresh six-class Skin training
Acne · Eczema · Psoriasis · Rosacea · Ringworm · Normal.

Use a **fresh GPU runtime** (Colab: Runtime → Change runtime type → GPU; Kaggle: Settings → Accelerator → GPU and Internet on).
The setup cell downloads the pinned public source automatically. No local upload, credentials, legacy Skin data or weights are needed.

The source dataset is pinned and verified; pre-augmented files are excluded before grouped splitting. Real accuracy is unknown until the final test cell runs.
Research/technical prototype only. Labels, upstream image rights and patient independence are not independently validated. Save/download your results before ending the session.

In [ ]:
from pathlib import Path
import os, sys, zipfile, subprocess, json, shutil
base = Path('/kaggle/working') if Path('/kaggle').exists() else Path('/content')
root = base / 'cliniq-six'
root.mkdir(parents=True, exist_ok=True)
revision = '7cbf3ed36897f4d845fd542e1b57428745c433ef'
git_env = dict(os.environ, GIT_LFS_SKIP_SMUDGE='1', GIT_TERMINAL_PROMPT='0')
if not (root / '.git').exists():
    subprocess.run(['git', 'init', str(root)], check=True)
    subprocess.run(['git', '-C', str(root), 'remote', 'add', 'origin', 'https://github.com/MohamedElsayedElbadwy/cliniq-ai-service.git'], check=True)
    subprocess.run(['git', '-C', str(root), 'fetch', '--depth', '1', 'origin', revision], check=True, env=git_env)
    subprocess.run(['git', '-C', str(root), 'checkout', '--detach', 'FETCH_HEAD'], check=True, env=git_env)
assert subprocess.check_output(['git', '-C', str(root), 'rev-parse', 'HEAD'], text=True).strip() == revision
os.chdir(root)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'skin/training/requirements-cloud.txt'], check=True)
print('Source ready:', root, revision)

In [ ]:
os.environ['KERAS_BACKEND'] = 'tensorflow'
import tensorflow as tf
assert tf.config.list_physical_devices('GPU'), 'Select a GPU runtime; restart if packages were previously imported.'
print('TensorFlow', tf.__version__, 'Keras', tf.keras.__version__)
print(tf.config.list_physical_devices('GPU'))
# For persistent storage on Colab, mount Drive yourself and point workspace to it.
workspace = root / 'workspaces' / 'skin-six'
def command(*parts):
    process = subprocess.Popen([sys.executable, *map(str, parts)], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    if process.wait():
        raise subprocess.CalledProcessError(process.returncode, process.args)

In [ ]:
if not workspace.exists():
    command('-m', 'skin.data.prepare_six', '--download', '--workspace', workspace)
from skin.training.six_class import load_protocol
protocol = load_protocol(workspace)
print(json.dumps(protocol['provenance'], indent=2))
assert protocol['provenance']['officialArchiveVerified']

## Train and choose using validation only
Default: EfficientNetB2, fresh ImageNet initialization, head training then fine-tuning.
Set `architectures` to both B0 and B2 for a validation comparison before finalization. Reduce batch size to 12 for limited GPU memory.
Existing run folders are protected. An interrupted run requires a new run name (keep the old checkpoint). Do not rerun training after viewing test results.

In [ ]:
architectures = ['efficientnetb2']  # Optional: ['efficientnetb0', 'efficientnetb2']
batch_size = 24
runs = []
for architecture in architectures:
    run = workspace / 'runs' / architecture
    if not (run / 'completed.json').exists():
        command('-m', 'skin.training.six_class', 'train', '--workspace', workspace, '--run', run,
                '--architecture', architecture, '--batch-size', batch_size)
    completed = json.loads((run / 'completed.json').read_text())
    runs.append((completed['validation']['macroF1'], run))
    print(architecture, completed['validation']['accuracy'], completed['validation']['macroF1'])
selected_run = max(runs, key=lambda item: item[0])[1]
print('Selected using validation macro-F1:', selected_run)

## Freeze selection, evaluate test, export
This locks the workspace to the chosen model and forbids further training on that split.
Temperature/abstention calibration uses validation only. Primary accuracy uses **all test images**.
Targets: accuracy ≥85%, macro-F1 ≥0.85 and recall ≥0.70 for every class. These are engineering targets, not promised results or clinical standards.
If targets fail the command exits nonzero and still saves the measured report. Review the data/errors; do not tune on the test set.

In [ ]:
export = workspace / 'export'
command('-m', 'skin.training.six_class', 'finalize', '--workspace', workspace,
        '--run', selected_run, '--output', export, '--batch-size', batch_size)
print((export / 'evaluation.md').read_text())

In [ ]:
# This cell can also be run after a quality-gate failure to inspect actual results.
assert (export / 'evaluation.json').exists(), 'Final evaluation must finish first.'
report = json.loads((export / 'evaluation.json').read_text())
print('Quality gate:', report['qualityGate'])
bundle = root / 'skin-six-results.zip'
with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as zf:
    for path in export.iterdir():
        if path.is_file(): zf.write(path, f'export/{path.name}')
    zf.write(workspace / 'provenance.json', 'provenance.json')
    for path in (workspace / 'splits').glob('*.csv'): zf.write(path, f'splits/{path.name}')
    for path in selected_run.glob('*.json'): zf.write(path, f'training/{path.name}')
print('Download:', bundle)
try:
    from google.colab import files
    files.download(str(bundle))
except ImportError:
    from IPython.display import FileLink, display
    display(FileLink(str(bundle)))
# After reviewing a passing evaluation, copy skin_six.keras + model_manifest.json
# + class_names.json to ClinIQ skin/artifacts/six-class/ and restart the service.